<a href="https://colab.research.google.com/github/marcehluna/VC/blob/main/Ejercicio_para_entrega_con_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Version 1.2 - Con Early Stopping (modificando el batch size) y entrenando el modelo con pesos de clase

## Descargar el Conjunto de Datos de Rostros

Descargar el archivo ZIP del conjunto de datos de emociones de rostros desde Kaggle, utilizando la URL proporcionada en el documento.


In [ ]:
get_ipython().system('wget -O human-face-emotions.zip "https://www.kaggle.com/api/v1/datasets/download/samithsachidanandan/human-face-emotions"')

In [ ]:
get_ipython().system('unzip -q human-face-emotions.zip -d human-face-emotions')

## Verificar Descompresión del Dataset

### Subtask:
Verificar la existencia y el contenido de la carpeta 'human-face-emotions' para confirmar que el archivo ZIP se ha descomprimido correctamente y que las imágenes están accesibles.


In [ ]:
print('Listing contents of human-face-emotions/Data/Angry directory (first 10 files):')
get_ipython().system('ls -F human-face-emotions/Data/Angry/ | head -n 10')

## Explorar Estructura del Dataset y Contar Imágenes

Inspeccionar la estructura de carpetas dentro de 'human-face-emotions/Data/', confirmar las categorías de emoción existentes y contar el número de imágenes en cada categoría para entender la distribución del dataset. Se informará el progreso y el resumen de la estructura.


In [ ]:
import os

base_path = 'human-face-emotions/Data/'

# Get a list of all emotion categories by listing the directories within base_path
emotion_categories = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

image_counts = {}
for category in emotion_categories:
    category_path = os.path.join(base_path, category)
    num_images = len([f for f in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, f))])
    image_counts[category] = num_images

print(f"Total number of emotion categories found: {len(emotion_categories)}")
print("Image counts per emotion category:")
for category, count in image_counts.items():
    print(f"- {category}: {count} images")

## Preparar Entorno para Preprocesamiento (OpenCV)

Instalar y/o importar las librerías necesarias, como OpenCV (cv2) y numpy, que serán utilizadas para el procesamiento de imágenes y la detección de rostros. Esto incluye la descarga del clasificador Haar Cascade para la detección de rostros.


In [ ]:
get_ipython().system('pip install opencv-python numpy')
print("OpenCV and NumPy libraries installed successfully.")

Descarga el Haar Cascade classifier


In [ ]:
import cv2
import numpy as np

print("OpenCV (cv2) and NumPy imported successfully.")

# Download the Haar Cascade classifier
get_ipython().system('wget -O haarcascade_frontalface_default.xml https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml')
print("Haar Cascade classifier downloaded successfully.")

## Cargar, Convertir a Escala de Grises y Redimensionar Imágenes

Implementar una función para cargar imágenes, convertirlas a escala de grises y redimensionarlas a una dimensión uniforme (por ejemplo, 48x48 píxeles, o una dimensión especificada por el usuario). Se aplicará esta función a todas las imágenes del dataset, mostrando un porcentaje de avance.


In [ ]:
def preprocess_image(image_path, target_size):
    # a. Use cv2.imread to load the image
    image = cv2.imread(image_path)

    # Check if image was loaded successfully
    if image is None:
        print(f"Warning: Could not load image from {image_path}. Skipping.")
        return None

    # b. Convert the loaded image to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # c. Resize the grayscale image to the target_size
    resized_image = cv2.resize(gray_image, target_size)

    # d. Return the processed image
    return resized_image

# 3. Initialize empty lists
processed_images = []
image_labels = []

# 4. Define target_image_size
target_image_size = (48, 48) # Example size, can be changed by user

print(f"Starting image preprocessing with target size: {target_image_size}")

# 5. Iterate through each emotion category
total_categories = len(emotion_categories)
for i, category in enumerate(emotion_categories):
    category_path = os.path.join(base_path, category)
    image_files = [f for f in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, f))]

    print(f"Processing category '{category}' ({i + 1}/{total_categories})... Total images: {len(image_files)}")

    for j, image_filename in enumerate(image_files):
        image_path = os.path.join(category_path, image_filename)

        # 9. Call the preprocess_image function
        processed_img = preprocess_image(image_path, target_image_size)

        if processed_img is not None:
            # 10. Append the processed image and label
            processed_images.append(processed_img)
            image_labels.append(category)

        # 11. Implement a progress tracking mechanism
        if (j + 1) % 1000 == 0: # Print progress every 1000 images
            print(f"  Processed {j + 1}/{len(image_files)} images in '{category}'")

print("\nImage preprocessing complete.")
print(f"Total processed images: {len(processed_images)}")
print(f"Total image labels: {len(image_labels)}")

# Convert lists to numpy arrays for further processing
processed_images = np.array(processed_images)
image_labels = np.array(image_labels)

print(f"Shape of processed_images array: {processed_images.shape}")
print(f"Shape of image_labels array: {image_labels.shape}")

Hay una distribucion dispar en la cantidad de caras de cada emocion. Esto podría condicionar la preparación del modelo.


## Detectar Rostros y Almacenar con Etiquetas

Utilizar el clasificador Haar Cascade para detectar rostros en cada imagen preprocesada. Solo los rostros detectados serán recortados y almacenados junto con su etiqueta emocional correspondiente en una estructura de datos adecuada (listas de imágenes y etiquetas). Se informará el progreso de la detección.


In [ ]:
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

detected_faces = []
detected_labels = []

print("Starting face detection...")

total_images_processed = len(processed_images)
for idx, (image, label) in enumerate(zip(processed_images, image_labels)):
    # Detect faces in the grayscale image
    faces = face_cascade.detectMultiScale(image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        for (x, y, w, h) in faces:
            # Crop the face from the image
            face_crop = image[y:y+h, x:x+w]
            # Resize the cropped face to the target_image_size
            resized_face = cv2.resize(face_crop, target_image_size)

            detected_faces.append(resized_face)
            detected_labels.append(label)

    if (idx + 1) % 10000 == 0: # Print progress every 10,000 images
        print(f"  Processed {idx + 1}/{total_images_processed} images. Detected {len(detected_faces)} faces so far.")

print("\nFace detection complete.")
print(f"Total processed images: {total_images_processed}")
print(f"Total detected faces: {len(detected_faces)}")

# Convert lists to numpy arrays
detected_faces = np.array(detected_faces)
detected_labels = np.array(detected_labels)

print(f"Shape of detected_faces array: {detected_faces.shape}")
print(f"Shape of detected_labels array: {detected_labels.shape}")

## Resumen del Preprocesamiento de Imágenes

Mostrar un resumen de la cantidad total de rostros detectados y almacenados, y la distribución de estos por categoría emocional, para confirmar que el proceso fue exitoso y que los datos están listos para las siguientes etapas.


In [ ]:
from collections import Counter

print(f"Total de rostros detectados y almacenados: {len(detected_faces)}")

# Calculate the distribution of detected faces by emotional category
emotion_distribution = Counter(detected_labels)

print("\nDistribución de rostros detectados por categoría emocional:")
for emotion, count in emotion_distribution.items():
    print(f"- {emotion}: {count} rostros")


## Normalizar Datos de Imágenes

Normalizar los valores de píxel de las imágenes en `detected_faces` para que estén en un rango adecuado (por ejemplo, 0 a 1), lo cual es crucial para el entrenamiento de redes neuronales.


In [ ]:
normalized_faces = detected_faces.astype('float32') / 255.0

print(f"Shape of normalized_faces array: {normalized_faces.shape}")
print(f"Minimum pixel value in normalized_faces: {normalized_faces.min()}")
print(f"Maximum pixel value in normalized_faces: {normalized_faces.max()}")

## Codificar Etiquetas Emocionales

Convertir las etiquetas emocionales (strings como 'Angry', 'Happy') a un formato numérico usando One-Hot Encoding, lo cual es necesario para la mayoría de los modelos de clasificación.


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 2. Create an instance of LabelEncoder and fit/transform the labels
label_encoder = LabelEncoder()
integer_encoded_labels = label_encoder.fit_transform(detected_labels)

print(f"Original labels (first 5): {detected_labels[:5]}")
print(f"Integer encoded labels (first 5): {integer_encoded_labels[:5]}")
print(f"Classes found by LabelEncoder: {label_encoder.classes_}")

# 3. Reshape integer_encoded_labels to be a 2D array
integer_encoded_labels_reshaped = integer_encoded_labels.reshape(len(integer_encoded_labels), 1)

# 4. Create an instance of OneHotEncoder and fit/transform the labels
one_hot_encoder = OneHotEncoder(sparse_output=False) # Ensure dense array output
one_hot_encoded_labels = one_hot_encoder.fit_transform(integer_encoded_labels_reshaped)

# 5. Print the shape of one_hot_encoded_labels and the first 5 rows
print(f"\nShape of one_hot_encoded_labels array: {one_hot_encoded_labels.shape}")
print("First 5 rows of one_hot_encoded_labels:\n", one_hot_encoded_labels[:5])

## Dividir el Conjunto de Datos

Dividir el conjunto de datos preprocesado (imágenes normalizadas y etiquetas codificadas) en conjuntos de entrenamiento, validación y prueba. El usuario debe poder especificar el porcentaje para cada conjunto.


In [ ]:
from sklearn.model_selection import train_test_split

# 2. Define the desired percentages for the training, validation, and test sets
train_size = 0.7
val_size = 0.15
test_size = 0.15

# Ensure the sum is 1 (or close enough due to floating point precision)
if not (train_size + val_size + test_size) == 1.0:
    print("Warning: train_size + val_size + test_size does not sum to 1.0. Adjusting test_size.")
    test_size = 1.0 - train_size - val_size

print(f"Splitting data with train_size={train_size}, val_size={val_size}, test_size={test_size}")

# 3. Split the data into a training set and a temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    normalized_faces, one_hot_encoded_labels, train_size=train_size, random_state=42
)

# 4. Calculate the relative size for the validation and test sets from the X_temp set
# X_temp now contains (val_size + test_size) proportion of the original dataset
val_size_relative = val_size / (val_size + test_size)

# 5. Split the X_temp and y_temp into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, train_size=val_size_relative, random_state=42
)

# 6. Print the shapes of the resulting sets
print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

## Preparar Datos para PCA

Redimensionar los conjuntos de datos de imágenes (X_train, X_val, X_test) de su formato 3D (muestras, altura, anchura) a un formato 2D (muestras, píxeles) para que sean compatibles con PCA.


In [ ]:
num_samples_train, height, width = X_train.shape
num_samples_val, _, _ = X_val.shape
num_samples_test, _, _ = X_test.shape

total_pixels = height * width

X_train_reshaped = X_train.reshape(num_samples_train, total_pixels)
X_val_reshaped = X_val.reshape(num_samples_val, total_pixels)
X_test_reshaped = X_test.reshape(num_samples_test, total_pixels)

print(f"Shape of X_train_reshaped: {X_train_reshaped.shape}")
print(f"Shape of X_val_reshaped: {X_val_reshaped.shape}")
print(f"Shape of X_test_reshaped: {X_test_reshaped.shape}")

In [ ]:
from sklearn.decomposition import PCA

# Initialize PCA to retain 95% of the variance
pca = PCA(n_components=0.95)

# Fit PCA only on the training data
X_train_pca = pca.fit_transform(X_train_reshaped)

# Transform the validation and test sets using the PCA fitted on the training data
X_val_pca = pca.transform(X_val_reshaped)
X_test_pca = pca.transform(X_test_reshaped)

print(f"Original number of features: {total_pixels}")
print(f"Number of components after PCA (retaining 95% variance): {pca.n_components_}")
print(f"Shape of X_train_pca: {X_train_pca.shape}")
print(f"Shape of X_val_pca: {X_val_pca.shape}")
print(f"Shape of X_test_pca: {X_test_pca.shape}")

## Definir el Modelo de Red Neuronal

Importar las librerías necesarias (como TensorFlow/Keras) y definir la arquitectura de la red neuronal, incluyendo las capas de entrada, ocultas y de salida, junto con sus funciones de activación.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print(f"TensorFlow Version: {tf.__version__}")
print("Sequential and Dense layers imported successfully.")

# 2. Create a Sequential Keras model
model = Sequential()

# 3. Add the input Dense layer
# input_shape should match the number of features after PCA (X_train_pca.shape[1])
model.add(Dense(256, activation='relu', input_shape=(X_train_pca.shape[1],)))

# 4. Add one or more additional hidden Dense layers
model.add(Dense(128, activation='relu'))

# 5. Add the output Dense layer
# The number of units should match the number of emotion classes (y_train.shape[1])
model.add(Dense(y_train.shape[1], activation='softmax'))

# 6. Show a summary of the model architecture
model.summary()


## Compilar el Modelo

Configurar el modelo para el entrenamiento especificando el optimizador, la función de pérdida y las métricas a utilizar.


In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Model compiled successfully with Adam optimizer, categorical_crossentropy loss, and accuracy metric.")

## Aplicar Ponderación de Clases para Mitigar el Desbalance

Calcular los pesos de clase para las etiquetas emocionales en el conjunto de entrenamiento (`y_train`) y aplicar estos pesos durante el entrenamiento del modelo para mitigar el impacto del desbalance de clases.

In [ ]:
from sklearn.utils import class_weight
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping # Importar EarlyStopping

# Convertir y_train de one-hot a etiquetas de clase para calcular los pesos
# Se necesita el `label_encoder` para obtener las clases en el orden correcto
y_train_labels = np.argmax(y_train, axis=1);

# Obtener las clases únicas del LabelEncoder
classes = label_encoder.classes_;

# Calcular los pesos de clase
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
);

# Convertir los pesos a un diccionario para Keras, mapeando los índices a los pesos
class_weights_dict = dict(enumerate(class_weights));

print("Pesos de clase calculados:");
for i, weight in class_weights_dict.items():
    print(f"- {classes[i]}: {weight:.2f}");

# Re-entrenar el modelo con los pesos de clase
epochs = 50;
batch_size = 64; # Se cambió el tamaño del lote a 64

# Definir Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitorear la pérdida de validación
    patience=10,         # Número de épocas sin mejora después de las cuales el entrenamiento se detendrá
    restore_best_weights=True # Restaurar los pesos del modelo de la mejor época
);

print(f"\nStarting model training with class weights for {epochs} epochs with batch size {batch_size}...");
print("Early Stopping está configurado para monitorear 'val_loss' con paciencia de 10 épocas.");

history_weighted = model.fit(
    X_train_pca, y_train,
    validation_data=(X_val_pca, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping], # Añadir el callback de Early Stopping
    class_weight=class_weights_dict, # Aplicar los pesos de clase
    verbose=1
);

print("\nModel training with class weights complete. History stored in 'history_weighted' variable.");

## Muestra las curvas de entrenamiento del modelo

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(history_weighted.history['accuracy'])
plt.plot(history_weighted.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(history_weighted.history['loss'])
plt.plot(history_weighted.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()


## Evaluar el Modelo

Evaluar el rendimiento del modelo entrenado utilizando el conjunto de datos de prueba (`X_test_pca`, `y_test`) para obtener métricas como la precisión y la pérdida.


In [ ]:
print("Evaluating the model on the test set...")

# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test_pca, y_test, verbose=0)

# Print the evaluation results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

## Mostrar Métricas de Clasificación por Clase

Calcular y mostrar la precisión, el recall y la puntuación F1 para cada clase de emoción utilizando el informe de clasificación de scikit-learn, para una evaluación detallada del rendimiento del modelo.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Realizar predicciones sobre el conjunto de prueba
y_pred_proba = model.predict(X_test_pca)
y_pred = np.argmax(y_pred_proba, axis=1)

# Convertir las etiquetas y_test de one-hot a etiquetas de clase para el informe
y_true = np.argmax(y_test, axis=1)

# Obtener las clases originales del LabelEncoder (asegurarse de que label_encoder esté disponible)
# Asumiendo que label_encoder se definió en una celda anterior y está en el ámbito
class_names = label_encoder.classes_

# Generar el informe de clasificación
report = classification_report(y_true, y_pred, target_names=class_names)

print("\nInforme de Clasificación por Clase:")
print(report)


## Generar y Visualizar la Matriz de Confusión


Generar una matriz de confusión para evaluar el rendimiento del modelo en el conjunto de prueba (`X_test_pca`, `y_test`), y visualizarla para entender mejor la distribución de aciertos y errores en la clasificación de emociones.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Realizar predicciones sobre el conjunto de prueba
y_pred_proba = model.predict(X_test_pca)
y_pred = np.argmax(y_pred_proba, axis=1)

# 2. Convertir las etiquetas y_test de one-hot a etiquetas de clase para la matriz de confusión
y_true = np.argmax(y_test, axis=1)

# Obtener las clases originales del LabelEncoder
# Asegurémonos de que label_encoder esté disponible del paso anterior
# Si no, tendríamos que re-crearlo o pasarlo
class_names = label_encoder.classes_

# 3. Calcular la matriz de confusión
cm = confusion_matrix(y_true, y_pred)

# 4. Visualizar la matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Etiqueta Predicha')
plt.ylabel('Etiqueta Verdadera')
plt.title('Matriz de Confusión')
plt.show()


## **Acá se carga la imagen del usuario para realizar la detección**

In [ ]:
from google.colab import files
import io
from PIL import Image
import cv2
import numpy as np

def predict_emotion_from_uploaded_image(image_path, target_size, face_cascade, pca_model, nn_model, label_encoder):
    # 1. Cargar la imagen
    uploaded_img = cv2.imread(image_path)

    if uploaded_img is None:
        return "Error al cargar la imagen. Asegúrate de que la ruta es correcta.", None, None

    # 2. Convertir a escala de grises
    gray_uploaded_img = cv2.cvtColor(uploaded_img, cv2.COLOR_BGR2GRAY)

    # 3. Detectar rostros
    faces = face_cascade.detectMultiScale(gray_uploaded_img, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) == 0:
        return "No se detectaron rostros en la imagen. Intenta con otra imagen.", None, None

    # Para simplificar, tomamos el rostro más grande si hay múltiples
    (x, y, w, h) = max(faces, key=lambda rect: rect[2] * rect[3])

    # Recortar y redimensionar el rostro detectado
    face_roi = gray_uploaded_img[y:y+h, x:x+w]
    face_resized = cv2.resize(face_roi, target_size)

    # 4. Normalizar la imagen del rostro
    normalized_face = face_resized.astype('float32') / 255.0

    # 5. Redimensionar para PCA (aplanar la imagen)
    # Asegurarse de que el número de píxeles coincida con el total_pixels del entrenamiento
    num_pixels = target_size[0] * target_size[1]
    flat_face = normalized_face.reshape(1, num_pixels) # Reshape a (1, num_pixels)

    # 6. Aplicar la transformación PCA
    # Asegurarse de usar el objeto PCA ya ajustado (pca)
    pca_transformed_face = pca_model.transform(flat_face)

    # 7. Realizar la predicción con el modelo de red neuronal
    predictions = nn_model.predict(pca_transformed_face)

    # 8. Interpretar el resultado
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    confidence = predictions[0][predicted_class_index]

    # Decodificar la etiqueta de la clase
    # Usar el label_encoder_original para obtener el nombre de la emoción
    predicted_emotion = label_encoder.inverse_transform([predicted_class_index])[0]

    return predicted_emotion, confidence, uploaded_img

# --- Proceso de Carga de Imagen por el Usuario y Predicción ---
print("Por favor, sube una imagen de un rostro para predecir su emoción:")

# Guardar la imagen cargada por el usuario
uploaded = files.upload()

if len(uploaded) == 0:
    print("No se ha subido ninguna imagen.")
else:
    for fn in uploaded.keys():
        print(f'Imagen "{fn}" cargada exitosamente.')
        # Guardar la imagen temporalmente para que cv2.imread pueda leerla
        with open(fn, 'wb') as f:
            f.write(uploaded[fn])

        # Realizar la predicción
        predicted_emotion, confidence, original_image = predict_emotion_from_uploaded_image(
            fn, target_image_size, face_cascade, pca, model, label_encoder
        )

        if predicted_emotion.startswith("Error") or predicted_emotion.startswith("No se detectaron"):
            print(predicted_emotion)
        else:
            print(f"Emoción Predicha: {predicted_emotion} con confianza de {confidence:.2f}")

            # Opcional: Mostrar la imagen con el rostro detectado y la predicción
            # Esto requiere dibujar el rectángulo del rostro en la imagen original
            # y luego mostrarla con matplotlib
            if original_image is not None:
                # Re-detectar caras en la imagen original (no en escala de grises) para dibujar
                faces_to_draw = face_cascade.detectMultiScale(cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY), scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
                if len(faces_to_draw) > 0:
                    (x, y, w, h) = max(faces_to_draw, key=lambda rect: rect[2] * rect[3])
                    cv2.rectangle(original_image, (x, y), (x+w, y+h), (255, 0, 0), 2) # Dibuja un rectángulo azul
                    text = f"{predicted_emotion}: {confidence:.2f}"
                    cv2.putText(original_image, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

                plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
                plt.title("Imagen Original con Predicción")
                plt.axis('off')
                plt.show()